In [ ]:
!ls "/content/drive/MyDrive/Papers/Elder Scam"

In [3]:
# ============================================================
# AEGIS — CELL 1: Mount Drive & Install
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets pandas

import os
SAVE_DIR = "/content/drive/MyDrive/Papers/Elder Scam"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")


# ============================================================
# AEGIS — CELL 2: Load ALL HuggingFace Datasets (No Auth)
# ============================================================

from datasets import load_dataset
import pandas as pd

hf_datasets = {}

# --- Dataset 1: Multi-agent scam conversations (1.6K rows) ---
# Best dataset. Multi-turn dialogues with 8 scam types + 8 victim personalities
# Columns: dialogue, personality, type, labels (0=legit, 1=scam)
print("1/5 Loading BothBosu/multi-agent-scam-conversation...")
ds = load_dataset("BothBosu/multi-agent-scam-conversation")
hf_datasets["multi_agent"] = pd.concat([ds["train"].to_pandas(), ds["test"].to_pandas()], ignore_index=True)
print(f"    ✓ {len(hf_datasets['multi_agent'])} rows | Columns: {list(hf_datasets['multi_agent'].columns)}")

# --- Dataset 2: Scam dialogue (1.6K rows) ---
# Simpler version — caller/receiver format
# Columns: dialogue, type, label (0=legit, 1=scam)
print("2/5 Loading BothBosu/scam-dialogue...")
ds = load_dataset("BothBosu/scam-dialogue")
hf_datasets["scam_dialogue"] = pd.concat([ds["train"].to_pandas(), ds["test"].to_pandas()], ignore_index=True)
print(f"    ✓ {len(hf_datasets['scam_dialogue'])} rows | Columns: {list(hf_datasets['scam_dialogue'].columns)}")

# --- Dataset 3: Single-agent scam conversations ---
# Same scam types but generated with single agent (Llama 3 70B)
# Columns: dialogue, personality, type, labels
print("3/5 Loading BothBosu/single-agent-scam-conversations...")
ds = load_dataset("BothBosu/single-agent-scam-conversations")
hf_datasets["single_agent"] = pd.concat([ds["train"].to_pandas(), ds["test"].to_pandas()], ignore_index=True)
print(f"    ✓ {len(hf_datasets['single_agent'])} rows | Columns: {list(hf_datasets['single_agent'].columns)}")

# --- Dataset 4: Scammer-Conversation (scam baiters + normal) ---
# Real-style scammer conversations plus normal ones
print("4/5 Loading BothBosu/Scammer-Conversation...")
try:
    ds = load_dataset("BothBosu/Scammer-Conversation")
    split_name = list(ds.keys())[0]  # get whatever split exists
    hf_datasets["scammer_convo"] = ds[split_name].to_pandas()
    print(f"    ✓ {len(hf_datasets['scammer_convo'])} rows | Columns: {list(hf_datasets['scammer_convo'].columns)}")
except Exception as e:
    print(f"    ⚠ Could not load: {e}")

# --- Dataset 5: All-scam-spam (1K rows, multilingual) ---
# Casual convos vs scam emails in ~10 languages
# Columns: text, is_scam
print("5/5 Loading FredZhang7/all-scam-spam...")
ds = load_dataset("FredZhang7/all-scam-spam")
split_name = list(ds.keys())[0]
hf_datasets["all_scam_spam"] = ds[split_name].to_pandas()
print(f"    ✓ {len(hf_datasets['all_scam_spam'])} rows | Columns: {list(hf_datasets['all_scam_spam'].columns)}")


# ============================================================
# AEGIS — CELL 3: Load GitHub Robocall Dataset (Real Calls!)
# ============================================================

# Real robocall transcripts from NC State University research
# 1432 actual robocall recordings transcribed via Whisper
print("\n6. Loading wspr-ncsu/robocall-audio-dataset from GitHub...")
try:
    robocall_url = "https://raw.githubusercontent.com/wspr-ncsu/robocall-audio-dataset/main/metadata.csv"
    hf_datasets["robocall_real"] = pd.read_csv(robocall_url)
    print(f"    ✓ {len(hf_datasets['robocall_real'])} rows | Columns: {list(hf_datasets['robocall_real'].columns)}")
except Exception as e:
    print(f"    ⚠ Could not load: {e}")


# ============================================================
# AEGIS — CELL 4: Preview Everything
# ============================================================

print("\n" + "=" * 70)
print("DATASET INVENTORY")
print("=" * 70)

for name, df in hf_datasets.items():
    print(f"\n{'─' * 50}")
    print(f"📦 {name} — {len(df)} rows")
    print(f"   Columns: {list(df.columns)}")

    # Show label distribution if available
    for col in ["labels", "label", "is_scam"]:
        if col in df.columns:
            dist = df[col].value_counts().to_dict()
            print(f"   Label distribution ({col}): {dist}")

    # Show scam types if available
    if "type" in df.columns:
        print(f"   Scam types: {df['type'].unique().tolist()}")

    # Show first dialogue snippet
    for col in ["dialogue", "text", "transcript"]:
        if col in df.columns:
            sample = str(df[col].iloc[0])[:150]
            print(f"   Sample: {sample}...")
            break

total = sum(len(df) for df in hf_datasets.values())
print(f"\n{'=' * 70}")
print(f"TOTAL: {total} rows across {len(hf_datasets)} datasets")
print(f"{'=' * 70}")


# ============================================================
# AEGIS — CELL 5: Unify Into Single Training Format
# ============================================================

unified = []

def add(dialogue, is_scam, scam_type="unknown", source="unknown"):
    """Add entry to unified dataset. Skip junk."""
    if not dialogue or len(str(dialogue).strip()) < 50:
        return
    unified.append({
        "dialogue": str(dialogue).strip(),
        "is_scam": int(is_scam) if int(is_scam) in [0, 1] else -1,
        "scam_type": str(scam_type),
        "source": source,
    })

# --- multi_agent: dialogue, personality, type, labels ---
if "multi_agent" in hf_datasets:
    for _, r in hf_datasets["multi_agent"].iterrows():
        add(r["dialogue"], r["labels"], r["type"], "hf_multi_agent")

# --- scam_dialogue: dialogue, type, label ---
if "scam_dialogue" in hf_datasets:
    for _, r in hf_datasets["scam_dialogue"].iterrows():
        add(r["dialogue"], r["label"], r["type"], "hf_scam_dialogue")

# --- single_agent: dialogue, personality, type, labels ---
if "single_agent" in hf_datasets:
    for _, r in hf_datasets["single_agent"].iterrows():
        add(r["dialogue"], r["labels"], r["type"], "hf_single_agent")

# --- scammer_convo: varies ---
if "scammer_convo" in hf_datasets:
    df = hf_datasets["scammer_convo"]
    # Detect columns adaptively
    dial_col = next((c for c in ["dialogue", "conversation", "text", "content"] if c in df.columns), None)
    label_col = next((c for c in ["labels", "label", "is_scam", "scam"] if c in df.columns), None)
    type_col = next((c for c in ["type", "category"] if c in df.columns), None)
    if dial_col:
        for _, r in df.iterrows():
            lbl = int(r[label_col]) if label_col else -1
            tp = str(r[type_col]) if type_col else "unknown"
            add(r[dial_col], lbl, tp, "hf_scammer_convo")

# --- all_scam_spam: text, is_scam ---
if "all_scam_spam" in hf_datasets:
    df = hf_datasets["all_scam_spam"]
    text_col = next((c for c in ["text", "content", "message"] if c in df.columns), None)
    label_col = next((c for c in ["is_scam", "label", "labels"] if c in df.columns), None)
    if text_col and label_col:
        for _, r in df.iterrows():
            add(r[text_col], r[label_col], "email_sms", "hf_all_scam_spam")

# --- robocall_real: transcript (all are real robocalls = scam) ---
if "robocall_real" in hf_datasets:
    df = hf_datasets["robocall_real"]
    if "transcript" in df.columns:
        for _, r in df.iterrows():
            add(r["transcript"], 1, "robocall", "github_robocall_real")

print(f"Unified entries: {len(unified)}")


# ============================================================
# AEGIS — CELL 6: Deduplicate & Split
# ============================================================

df_all = pd.DataFrame(unified)

# Dedup on dialogue text
before = len(df_all)
df_all = df_all.drop_duplicates(subset=["dialogue"], keep="first").reset_index(drop=True)
print(f"Deduplication: {before} → {len(df_all)}")

# Split by label
labeled = df_all[df_all["is_scam"] >= 0]
unlabeled = df_all[df_all["is_scam"] < 0]

scam = labeled[labeled["is_scam"] == 1]
legit = labeled[labeled["is_scam"] == 0]

print(f"\nLabeled:   {len(labeled)}")
print(f"  Scam:    {len(scam)}")
print(f"  Legit:   {len(legit)}")
print(f"Unlabeled: {len(unlabeled)}")
print(f"\nSources:   {labeled['source'].value_counts().to_dict()}")
print(f"Scam types: {labeled['scam_type'].value_counts().to_dict()}")

# Average dialogue length
print(f"\nAvg dialogue length: {labeled['dialogue'].str.len().mean():.0f} chars")
print(f"Min: {labeled['dialogue'].str.len().min()} | Max: {labeled['dialogue'].str.len().max()}")


# ============================================================
# AEGIS — CELL 7: Save Everything to Google Drive
# ============================================================

import json

# Save unified JSONL files
scam.to_json(f"{SAVE_DIR}/scam_conversations.jsonl", orient="records", lines=True)
legit.to_json(f"{SAVE_DIR}/legit_conversations.jsonl", orient="records", lines=True)
df_all.to_json(f"{SAVE_DIR}/all_conversations.jsonl", orient="records", lines=True)

# Also save raw HuggingFace DataFrames as CSVs for reference
raw_dir = f"{SAVE_DIR}/raw_datasets"
os.makedirs(raw_dir, exist_ok=True)
for name, df in hf_datasets.items():
    df.to_csv(f"{raw_dir}/{name}.csv", index=False)

# Save a metadata summary
meta = {
    "total_unified": len(df_all),
    "total_scam": len(scam),
    "total_legit": len(legit),
    "total_unlabeled": len(unlabeled),
    "sources": labeled["source"].value_counts().to_dict(),
    "scam_types": labeled["scam_type"].value_counts().to_dict(),
    "datasets_loaded": list(hf_datasets.keys()),
}
with open(f"{SAVE_DIR}/dataset_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"\n✅ All data saved to: {SAVE_DIR}")
print(f"\nFiles:")
for f in sorted(os.listdir(SAVE_DIR)):
    path = os.path.join(SAVE_DIR, f)
    if os.path.isfile(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"  {f} ({size_kb:.0f} KB)")
    else:
        print(f"  {f}/ (directory)")

print(f"\n🎯 Ready for Step 2: Auto-annotation with Gemma 4 26B")
print(f"   scam_conversations.jsonl → {len(scam)} conversations to annotate with epistemic vectors")
print(f"   legit_conversations.jsonl → {len(legit)} conversations for contrast training")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save directory: /content/drive/MyDrive/Papers/Elder Scam
1/5 Loading BothBosu/multi-agent-scam-conversation...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

agent_conversation_train.csv: 0.00B [00:00, ?B/s]

agent_conversation_test.csv: 0.00B [00:00, ?B/s]

...

    ✓ 1000 rows | Columns: ['conversation', 'label']
5/5 Loading FredZhang7/all-scam-spam...


README.md: 0.00B [00:00, ?B/s]

junkmail_dataset.csv:   0%|          | 0.00/67.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/42619 [00:00<?, ? examples/s]

    ✓ 42619 rows | Columns: ['text', 'is_spam']

6. Loading wspr-ncsu/robocall-audio-dataset from GitHub...
    ✓ 1432 rows | Columns: ['file_name', 'language', 'transcript', 'case_details', 'case_pdf']

DATASET INVENTORY

──────────────────────────────────────────────────
📦 multi_agent — 1600 rows
   Columns: ['dialogue', 'personality', 'type', 'labels']
   Label distribution (labels): {0: 800, 1: 800}
   Scam types: ['appointment', 'delivery', 'insurance', 'refund', 'reward', 'ssn', 'support', 'wrong']
   Sample: Innocent: Hello.  Suspect: Hi, this is Karen from Smith & Co. I'm calling to confirm your appointment with Dr. Lee on Thursday at 2 PM. Can you please...

──────────────────────────────────────────────────
📦 scam_dialogue — 1600 rows
   Columns: ['dialogue', 'type', 'label']
   Label distribution (label): {1: 800, 0: 800}
   Scam types: ['ssn', 'refund', 'support', 'reward', 'delivery', 'insurance', 'telemarketing', 'wrong']
   Sample: caller: Hello, is this John? receiver: 